# SQL Data Retrieval and Validation

## Objective
Connect Python to a PostgreSQL database containing 24,386,900 credit card transactions. Retrieve selected records using SQL and validate the results against previous Pandas analysis.

## Tools
- PostgreSQL
- SQL
- Python
- Pandas
- SQLAlchemy

## Business Value
Demonstrate efficient data retrieval and validation of banking transaction data without unnecessarily loading the entire database into memory.

In [ ]:
%pip install sqlalchemy "psycopg[binary]"

In [ ]:
import pandas as pd
import sqlalchemy
import psycopg

print("Pandas:", pd.__version__)
print("SQLAlchemy:", sqlalchemy.__version__)
print("Psycopg:", psycopg.__version__)

print("Environment ready!")

In [ ]:
import os
from getpass import getpass
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Configure your own local PostgreSQL connection using environment variables.
# Never commit passwords or connection secrets to GitHub.
url = URL.create(
    drivername="postgresql+psycopg",
    username=os.getenv("PGUSER", "fraud_analyst"),
    password=os.getenv("PGPASSWORD") or getpass("PostgreSQL password: "),
    host=os.getenv("PGHOST", "localhost"),
    port=int(os.getenv("PGPORT", "5432")),
    database=os.getenv("PGDATABASE", "banking"),
)
engine = create_engine(url)
with engine.connect() as connection:
    print("Database connection successful:", connection.execute(text("SELECT 1")).scalar())

In [ ]:
import pandas as pd

query = """
SELECT
    user_id,
    card_id,
    amount,
    use_chip,
    is_fraud
FROM fraud_transaction
WHERE is_fraud = TRUE
LIMIT 10;
"""

df_sql = pd.read_sql(query, engine)

df_sql

### Database Retrieval Validation

Successfully established a read-only connection between Python and PostgreSQL using SQLAlchemy and psycopg.

Retrieved 10 transaction records and five selected columns into a Pandas DataFrame. This demonstrates selective data retrieval without loading the entire 24,386,900-row dataset into Python memory.

**Observation:** The transaction amount is stored as text containing currency symbols and will require numeric conversion before financial analysis.

In [ ]:
query = """
SELECT COUNT(*) FROM fraud_transaction
WHERE is_fraud =TRUE;
"""

df_fraud_count = pd.read_sql(query, engine)

df_fraud_count

In [ ]:
query = """
-- Write your SQL here
SELECT COUNT(*) AS total_transactions,
        COUNT(*) FILTER (WHERE is_fraud=TRUE) AS fraud_transactions from fraud_transaction;

"""

df_fraud_rate = pd.read_sql(query, engine)

df_fraud_rate

In [ ]:
df_fraud_rate["fraud_rate"]=(100*df_fraud_rate["fraud_transactions"]/df_fraud_rate["total_transactions"])
df_fraud_rate

### Finding 1: Overall Fraud Prevalence

The dataset contains **24,386,900 credit card transactions**, of which **29,757 are labeled fraudulent**, resulting in an overall fraud rate of approximately **0.122%**.

SQL was used to aggregate transaction counts directly in PostgreSQL. Python and Pandas were then used to calculate the fraud percentage.

**Business implication:** Fraudulent transactions represent a very small proportion of the dataset. This class imbalance must be considered when developing predictive models. Accuracy alone may be misleading, so precision, recall, and precision-recall analysis will be important evaluation measures.

In [ ]:
query = """
SELECT
    use_chip,
    COUNT(*) AS fraud_count
FROM fraud_transaction
WHERE is_fraud =TRUE
GROUP BY use_chip;
"""

df_fraud_method = pd.read_sql(query, engine)

df_fraud_method

In [ ]:
query = """
SELECT
    use_chip,
    COUNT(*) AS total_transactions,
    COUNT(*) FILTER (WHERE is_fraud = TRUE) AS fraud_transactions
FROM fraud_transaction
GROUP BY use_chip;
"""

df_method = pd.read_sql(query, engine)
df_method

In [ ]:
df_method["fraud_rate"] = (
    100 * df_method["fraud_transactions"]
    / df_method["total_transactions"]
)
df_method

In [ ]:
df_method.sort_values(by="fraud_rate", ascending=False)

## Fraud Rate Analysis by Transaction Method

### Business Objective
Determine which transaction method—Online, Chip, or Swipe—has the highest observed fraud rate.

### Methodology
PostgreSQL was used to aggregate the total number of transactions and fraudulent transactions by payment method. The summarized data was imported into Python using `pd.read_sql()`, and Pandas was used to calculate fraud rates.

Fraud Rate (%) = (Fraudulent Transactions / Total Transactions) × 100

### Results

| Transaction Method | Total Transactions | Fraudulent Transactions | Fraud Rate |
|---|---:|---:|---:|
| Online | 2,713,220 | 18,349 | 0.677% |
| Chip | 6,287,598 | 4,836 | 0.077% |
| Swipe | 15,386,082 | 6,572 | 0.043% |

### Key Findings

1. Online transactions had the highest observed fraud rate at approximately 0.677%.
2. Online transactions had approximately 8.8 times the fraud rate of chip transactions and 15.8 times the fraud rate of swipe transactions.
3. Although swipe transactions represented the largest transaction volume, they had the lowest observed fraud rate.
4. Comparing fraud rates rather than fraud counts provides a more meaningful assessment of relative risk because it accounts for differences in transaction volume.

### Business Implications

The findings suggest that online transactions warrant greater attention in fraud risk monitoring.

However, a higher observed fraud rate does not justify automatically declining all online transactions. Such a policy could reject large numbers of legitimate purchases and create unnecessary financial and customer-service costs.

Further analysis should investigate transaction amounts, merchant categories, geographic patterns, and other risk indicators before recommending specific fraud prevention strategies.

### Limitations

This analysis is descriptive and does not establish that transaction method causes fraud. The observed differences may be influenced by other transaction characteristics.

### Conclusion

Online transactions represent the highest observed fraud risk among the three payment methods in this dataset. The results provide a foundation for further fraud risk modeling and evaluation of targeted detection strategies.

In [ ]:
query="""
SELECT
    is_fraud,
    COUNT(*) AS transaction_count,
    AVG(REPLACE(amount, '$', '')::NUMERIC) AS avg_amount
FROM fraud_transaction
GROUP BY is_fraud;
"""
df_amount = pd.read_sql(query,engine)
df_amount

In [ ]:
# Extract average fraudulent transaction amount
fraud_avg = df_amount["avg_amount"].iloc[1]

# Extract average legitimate transaction amount
legit_avg = df_amount["avg_amount"].iloc[0]

# Calculate the ratio
ratio = fraud_avg / legit_avg

# Display the result
round(ratio,2)


## Python Analysis: Comparing Average Transaction Amounts

### Objective

Calculate how many times larger the average fraudulent transaction amount is compared with the average legitimate transaction amount.

### Methodology

Using the SQL results imported into a Pandas DataFrame (`df_amount`), Python was used to:

1. Extract the average fraudulent transaction amount using `.iloc[1]`.
2. Extract the average legitimate transaction amount using `.iloc[0]`.
3. Divide the fraudulent average by the legitimate average to calculate the ratio.

The row positions were verified against the displayed DataFrame.

### Results

| Transaction Type | Average Amount |
|---|---:|
| Fraudulent | $108.59 |
| Legitimate | $43.55 |

**Ratio = 108.59 / 43.55 ≈ 2.49**

### Business Interpretation

Fraudulent transactions have an average amount approximately **2.49 times higher** than legitimate transactions.

This suggests that transaction amount may be an important feature when identifying potentially fraudulent activity.

However, a higher average does not mean every large transaction is fraudulent. Additional analysis is required to understand the distribution of transaction amounts and the overlap between fraudulent and legitimate transactions.

### Key Takeaway

Transaction amount shows a meaningful difference between fraudulent and legitimate transactions, making it a promising variable for further fraud risk analysis.

**Next Step:** Investigate fraud rates across different transaction amount ranges to determine whether higher transaction amounts are associated with greater fraud risk.

In [ ]:
query = """
SELECT
    COUNT(*) AS legitimate_above_100
FROM fraud_transaction
WHERE is_fraud = FALSE
  AND REPLACE(amount, '$', '')::NUMERIC > 100;
"""

df_threshold = pd.read_sql(query, engine)

df_threshold

In [ ]:
false_positive_rate = (
    df_threshold["legitimate_above_100"].iloc[0]
    / df_amount["transaction_count"].iloc[0]
) * 100

round(false_positive_rate,2)